# MRP Engine — Cow Runner

This notebook runs the Cow MRP (dairy cow life-cycle model: state = lactation, DIM, DP) through the generic MRP engine.

**How to use:**
1. Keep `DOMAIN = 'cow'` in Cell 3
2. Run all cells top-to-bottom

**Architecture:**
```
mrp/core.py          ← shared generic engine (Bellman, distributions, etc.)
mrp/engine.py        ← spec orchestration / validation
cow/                 ← cow biology domain
mrp_engine.ipynb     ← this notebook (just a runner)
```

In [ ]:
import os
import sys
# project root (mrp/, cow/) is one level up; exports are written to outputs/notebook/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
OUT_DIR = os.path.join(ROOT, 'outputs', 'notebook')
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(OUT_DIR)

# ------------------------------------------------------------
# Cache-bust: unlike every other Runner script (farm_generator.py /
# farm_simulator.py / benchmark_worker.py), which each start a FRESH
# `python script.py` process every run, this notebook keeps ONE
# long-lived kernel across "Run All". Python caches every imported
# module in sys.modules by name, so re-running this cell's imports in
# an ALREADY-RUNNING kernel does NOT re-execute cow/parameters.py (or
# anything that already imported it) even after you edit TIME_UNIT
# there -- you'd silently keep running the OLD resolution. Purging the
# relevant modules here, every time this cell runs, makes "Run All"
# (with or without a kernel restart) always re-read the CURRENT
# contents of cow/parameters.py -- the one authoritative time-
# resolution source stays authoritative across reruns, not just on a
# fresh kernel.
# ------------------------------------------------------------
for _mod_name in list(sys.modules):
    if (_mod_name == 'cow' or _mod_name.startswith('cow.')
            or _mod_name == 'mrp' or _mod_name.startswith('mrp.')):
        del sys.modules[_mod_name]

import numpy as np
import pandas as pd

# Generic MRP engine (shared by all domains)
from mrp.core import (
    annual_to_per_step_gamma,
    assemble_matrix, validate_row_stochasticity,
    get_transitions,
    evaluate_finite_horizon, evaluate_infinite_horizon,
    make_initial_distribution, distribution_recursive,
    distribution_matrix_power, distribution_eigen,
    distribution_recursive_origin_tracked,
    steps_per_year, export_transitions, export_rewards,
)
from mrp.engine import build_states
from scipy.sparse import save_npz

print("mrp (core/engine) imported successfully.")

## Step 0: Set Parameters

`DOMAIN` stays `'cow'` (the only domain in this fork).

In [ ]:
import os

# ----------------------------------------------------------
# DOMAIN -- Cow only in this fork
# ----------------------------------------------------------
DOMAIN = 'cow'

# ----------------------------------------------------------
# USER SETTINGS (MRP evaluation parameters)
# ----------------------------------------------------------
GAMMA_ANNUAL = 0.95      # 5% annual discount rate
NUM_YEARS = 5            # How many years to evaluate

# ----------------------------------------------------------
# Import domain-specific modules
# ----------------------------------------------------------
from cow.parameters import TIME_UNIT, UNIT_DAYS, MAX_DIM, MAX_DP, LAST_BREEDING_DIM, MILK_FLOOR_KG
from cow.states import variable_domains, derived_variables
from cow.probabilities import compute_probabilities, probability_specs
from cow.rewards import reward_specs, compute_rewards
from cow.transitions import compute_transitions, get_replacement_state_id

INITIAL_STATE = {'lactation': 1, 'dim': 0, 'dp': 0}
print("Domain: COW (life-cycle model)")
gamma = annual_to_per_step_gamma(GAMMA_ANNUAL, UNIT_DAYS)

# Compute per-step gamma. TIME_UNIT/UNIT_DAYS come from this domain's own
# parameters module -- the ONE authoritative time-resolution source per
# domain. Nothing below, or anywhere else in this notebook, redefines
# time resolution independently; every later step (states, probabilities,
# transitions, rewards, horizon evaluation, simulation, replacement
# tracking, plotting) reads UNIT_DAYS/TIME_UNIT from here, directly or
# transitively.
print("=" * 60)
print("TIME RESOLUTION (single source of truth: this domain's own parameters module)")
print("=" * 60)
print(f"TIME_UNIT  = {TIME_UNIT!r}   (1 MRP step = {UNIT_DAYS} real day(s))")
_env_override = os.environ.get("COW_TIME_UNIT")
if _env_override is not None:
    print(f"NOTE: the COW_TIME_UNIT environment variable is set to {_env_override!r} in this "
          f"process -- that value, not the hardcoded default inside cow/parameters.py, is what "
          f"actually won (cow/parameters.py's own documented precedence). If TIME_UNIT above "
          f"isn't what you just edited in the file, this env var is why -- unset/change it, or "
          f"edit its value here, not the file, to switch resolution.")
print(f"MAX_DIM    = {MAX_DIM} steps, MAX_DP = {MAX_DP} steps  (state-space time axes, in {TIME_UNIT}s)")
print(f"Annual discount {GAMMA_ANNUAL} -> per-step gamma {gamma:.6f}")
print(f"{steps_per_year(UNIT_DAYS)} steps/year at this resolution")

## Step 1: Build State Space

In [ ]:
print("=" * 60)
print(f"STEP 1: Building state space ({DOMAIN})")
print("=" * 60)

from cow.states import build_cow_states
states = build_cow_states()   # pruned: only reachable (lactation, dim, dp) states
print(f"Total states: {len(states):,}")
print(f"\nState columns: {list(states.columns)}")
print(f"\nFirst 10 states:")
states.head(10)

## Step 2: Compute Probabilities

In [ ]:
print("=" * 60)
print(f"STEP 2: Computing probabilities ({DOMAIN})")
print("=" * 60)

print("\nDeclared probability specs:")
for spec in probability_specs:
    deps = str(spec['depends_on'])
    print(f"  {spec['name']:25s} depends on {deps:35s} active: {spec['method']}")

probs = compute_probabilities(states)
print(f"\nProbability arrays computed: {list(probs.keys())}")

# Quick sanity check
sample = states.head(5).copy()
for k in list(probs.keys())[:3]:
    sample[k] = probs[k][:5]
sample

## Step 3: Build Transition Matrix

In [ ]:
print("=" * 60)
print(f"STEP 3: Building transition matrix ({DOMAIN})")
print("=" * 60)

rows, cols, vals = compute_transitions(states, probs)
matrix = assemble_matrix(states, rows, cols, vals)
validate_row_stochasticity(matrix, states)
print(f"Nonzero entries: {matrix.nnz:,}")
print(f"Matrix shape: {matrix.shape}")
print(f"Sparsity: {matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100:.2f}%")

## Step 4: Compute Reward Vector

In [ ]:
print("=" * 60)
print(f"STEP 4: Computing reward vector ({DOMAIN})")
print("=" * 60)

# Uses the sanctioned cow.rewards.compute_rewards(states, probs) entry
# point rather than calling run_specs(reward_specs, ...) directly, since
# reward_specs now includes the environmental reward specs (nitrogen,
# enteric methane, ...) which depend on model_output quantities
# (manure_n_g, enteric_ch4_g, ...) produced by the staged
# compute_environment() biological layer -- compute_rewards() builds
# that merged context internally (see cow/rewards/__init__.py).
rewards = compute_rewards(states, probs)
print(f"Reward components: {list(rewards.keys())}")
print(f"Total reward range: [{rewards['total'].min():.2f}, {rewards['total'].max():.2f}]")

print(f"\nReward specs declared:")
for spec in reward_specs:
    deps = str(spec['depends_on'])
    print(f"  {spec['name']:25s} depends on {deps:35s} active: {spec['method']:8s} sign: {'+' if spec['sign']>0 else '-'}")

# Show breakdown for first few states
reward_df = pd.DataFrame({k: v[:5] for k, v in rewards.items()})
reward_df

## Step 5: Finite-Horizon Bellman Evaluation

In [ ]:
print("=" * 60)
print("STEP 5: Finite-horizon Bellman evaluation")
print("=" * 60)

V_finite = evaluate_finite_horizon(matrix, rewards['total'], gamma, NUM_YEARS, UNIT_DAYS)

## Step 6: Infinite-Horizon Exact Solve

In [ ]:
print("=" * 60)
print("STEP 6: Infinite-horizon exact solve")
print("=" * 60)

V_infinite = evaluate_infinite_horizon(matrix, rewards['total'], gamma)
print(f"mean V = {V_infinite.mean():.2f}  min = {V_infinite.min():.2f}  max = {V_infinite.max():.2f}")

# Show top 10 most valuable states
top_states = states.copy()
top_states['V_infinite'] = V_infinite
top_states.nlargest(10, 'V_infinite')

## Step 7: State Distribution Evolution (3 Methods)

In [ ]:
print("=" * 60)
print("STEP 7: State distribution evolution")
print("=" * 60)

pi0 = make_initial_distribution(states, **INITIAL_STATE)
print(f"Initial distribution: 1.0 mass at state {np.where(pi0 == 1)[0][0]}")

num_steps = steps_per_year(UNIT_DAYS)
print(f"\nEvolving {num_steps} steps ahead (1 year)...")

# Method 1: Recursive
pi_recursive = distribution_recursive(matrix, pi0, num_steps)
exp_reward_recursive = pi_recursive @ rewards['total']
print(f"Method 1 (recursive):      sum = {pi_recursive.sum():.6f}  expected reward = {exp_reward_recursive:.2f}")

# Method 2 & 3: only for small state spaces
if len(states) <= 3000:
    pi_power = distribution_matrix_power(matrix, pi0, num_steps)
    exp_reward_power = pi_power @ rewards['total']
    print(f"Method 2 (matrix power):   sum = {pi_power.sum():.6f}  expected reward = {exp_reward_power:.2f}")

    pi_eigen = distribution_eigen(matrix, pi0, num_steps)
    exp_reward_eigen = pi_eigen @ rewards['total']
    print(f"Method 3 (eigen):          sum = {pi_eigen.sum():.6f}  expected reward = {exp_reward_eigen:.2f}")
else:
    print(f"Skipping Methods 2 & 3: {len(states):,} states is too many for dense matrix operations.")
    print("  (Switch to coarser TIME_UNIT or a smaller domain to run them.)")

## Step 7b: Replacement-Aware Distribution Evolution

Splits the Step 7 expected-distribution evolution into "never replaced
since t=0" vs. "replaced (death/culling/max-lactation) at least once"
mass, using the domain's own replacement target
(`cow.transitions.get_replacement_state_id`) -- the exact same mechanism
`farm_simulator.py` uses, not a second definition of
replacement. Timing is expressed in the current `TIME_UNIT` throughout.

In [ ]:
print("=" * 60)
print("STEP 7b: Replacement-aware distribution evolution")
print("=" * 60)

# Same generic mechanism farm_simulator.py uses --
# mrp_core.distribution_recursive_origin_tracked partitions the
# evolving expected distribution into "never replaced since t=0"
# vs "replaced (death/culling/max-lactation) at least once" mass,
# using the domain's OWN replacement target
# (cow.transitions.get_replacement_state_id) -- not a second,
# independently-defined notion of replacement.
replacement_id = get_replacement_state_id(states)
_repl_row = states.iloc[replacement_id][[c for c in states.columns if c != 'state_id']].to_dict()
print(f"Replacement state id: {replacement_id} -> {_repl_row}")

pi_orig, pi_tag = pi0.copy(), np.zeros_like(pi0)
orig_trajectory = [pi_orig.copy()]
tagged_trajectory = [pi_tag.copy()]
for _ in range(num_steps):
    pi_orig, pi_tag = distribution_recursive_origin_tracked(matrix, pi_orig, pi_tag, 1, replacement_id)
    orig_trajectory.append(pi_orig.copy())
    tagged_trajectory.append(pi_tag.copy())

# Regular + replacement must reconstruct the undivided distribution
# exactly, at every step (same invariant farm_simulator.py asserts).
for step, (po, pt) in enumerate(zip(orig_trajectory, tagged_trajectory)):
    assert np.isclose((po + pt).sum(), 1.0, atol=1e-9), f"Mass not conserved at step {step}"

replaced_fraction = [pt.sum() for pt in tagged_trajectory]
print(f"\nFraction of the herd replaced at least once, by {TIME_UNIT} (1 MRP step = 1 {TIME_UNIT}):")
stride = max(1, num_steps // 10)
for step in range(0, num_steps + 1, stride):
    print(f"  {TIME_UNIT.capitalize()} {step:3d}: {replaced_fraction[step]*100:5.1f}% replaced")

## Step 8: Export Results

In [ ]:
print("=" * 60)
print("STEP 8: Exporting results")
print("=" * 60)

save_npz(f'transition_matrix_{DOMAIN}.npz', matrix)
print(f"Saved: transition_matrix_{DOMAIN}.npz")

# unit_prefix='u' -- resolution-neutral (same convention benchmark_worker.py
# already uses), NOT a fixed 'm' -- exported dim/dp columns must never claim
# a specific resolution (e.g. stale "mim"/"mp") when TIME_UNIT is actually
# 'day' or 'week'. See cow/parameters.py's TIME_UNIT/UNIT_DAYS single
# source of truth.
transitions_long = export_transitions(matrix, states, unit_prefix='u')
transitions_long.to_parquet(f'transition_matrix_{DOMAIN}_long.parquet', index=False)
print(f"Saved: transition_matrix_{DOMAIN}_long.parquet ({len(transitions_long):,} rows)")

print("\nFirst 15 rows of transition export (grouped by source state):")
print(transitions_long.head(15).to_string())

reward_df = export_rewards(states, rewards, unit_prefix='u')
reward_df.to_parquet(f'reward_vector_{DOMAIN}.parquet', index=False)
print(f"\nSaved: reward_vector_{DOMAIN}.parquet ({len(reward_df):,} rows)")

if len(transitions_long) <= 200000:
    transitions_long.to_csv(f'transition_matrix_{DOMAIN}_long.csv', index=False)
    reward_df.to_csv(f'reward_vector_{DOMAIN}.csv', index=False)
    print(f"Saved: transition_matrix_{DOMAIN}_long.csv and reward_vector_{DOMAIN}.csv")
else:
    print("CSV export skipped (too many rows)")

print("\n" + "=" * 60)
print(f"DONE! ({DOMAIN} domain)")
print("=" * 60)

## Step 9: Stochastic Trajectory Simulation

This step demonstrates the domain-agnostic stochastic simulation methods
added to `mrp_core.py`:

- **Single trajectory**: `simulate_stochastic_trajectory()` — sample one possible
  future path for N entities starting from the same initial state.
- **Monte Carlo**: `simulate_monte_carlo()` — run N independent simulations to
  estimate confidence intervals.

These work for any domain; here, `DOMAIN = 'cow'`.

In [ ]:
from mrp.core import simulate_stochastic_trajectory

print("=" * 60)
print("STEP 9a: Single Stochastic Trajectory")
print("=" * 60)

# Start with 100 entities all in the initial state
n_entities = 100
initial_state_id = np.where(pi0 == 1)[0][0]
initial_state_ids = np.full(n_entities, initial_state_id)

n_steps = steps_per_year(UNIT_DAYS)  # 1 year

history = simulate_stochastic_trajectory(matrix, initial_state_ids, n_steps, seed=42)
print(f"History shape: {history.shape}  (steps+1, entities)")
print(f"\nFirst 5 entities' state IDs over time:")
print(history[:, :5])

# Show how many entities are in each state at each step
print(f"\nState diversity over time:")
for step in range(0, n_steps + 1, 3):
    unique, counts = np.unique(history[step], return_counts=True)
    print(f"  Step {step:3d}: {len(unique)} distinct states")

In [ ]:
from mrp.core import simulate_monte_carlo

print("=" * 60)
print("STEP 9b: Monte Carlo Simulation")
print("=" * 60)

n_sims = 50
n_entities = 100
initial_state_id = np.where(pi0 == 1)[0][0]
initial_state_ids = np.full(n_entities, initial_state_id)

n_steps = steps_per_year(UNIT_DAYS)

print(f"Running {n_sims} simulations with {n_entities} entities each...")
trajectories = simulate_monte_carlo(matrix, initial_state_ids, n_steps, n_sims=n_sims, seed=42)
print(f"Returned {len(trajectories)} trajectories")

# Track how many entities end up in the initial state at final step
final_in_initial = []
for traj in trajectories:
    count = np.sum(traj[-1] == initial_state_id)
    final_in_initial.append(count)

print(f"\nFinal step: entities still in initial state")
print(f"  Mean: {np.mean(final_in_initial):.1f}")
print(f"  Std:  {np.std(final_in_initial):.1f}")
print(f"  Min:  {np.min(final_in_initial)}")
print(f"  Max:  {np.max(final_in_initial)}")

In [ ]:
# Domain-aware grouping for Monte Carlo results
# Cow: group by reproductive flag

print("=" * 60)
print("STEP 9c: Monte Carlo Grouped Summary")
print("=" * 60)

from cow.states import repro_status
def label_fn(row):
    return str(repro_status(row['dp']))
group_name = "Reproductive status (from dp)"

# Summarize across all simulations
all_labels = set()
for traj in trajectories:
    for step in range(n_steps + 1):
        labels = [label_fn(states.iloc[sid]) for sid in traj[step]]
        all_labels.update(labels)
all_labels = sorted(all_labels)

print(f"Tracking {len(all_labels)} labels: {all_labels}")
print(f"\nMean counts per label at final step (across {n_sims} sims):")

for label in all_labels:
    counts = []
    for traj in trajectories:
        labels = [label_fn(states.iloc[sid]) for sid in traj[-1]]
        counts.append(labels.count(label))
    print(f"  {label:20s}: mean={np.mean(counts):6.1f}  std={np.std(counts):5.1f}  "
          f"p05={np.percentile(counts,5):5.0f}  p95={np.percentile(counts,95):5.0f}")

## Step 10: Interactive Plotly Visualization

X-axis, titles, and hover labels all read the current `TIME_UNIT` --
switching `cow/parameters.py`'s `TIME_UNIT` and rerunning from the top
changes what these plots call one time step ("Day"/"Week"/"Month")
*and* how many steps make up the plotted year, not just the label text.

In [ ]:
print("=" * 60)
print("STEP 10: Interactive Plotly visualization")
print("=" * 60)

try:
    import plotly.graph_objects as go
except ImportError:
    print("WARNING: plotly not installed (pip install plotly) -- skipping this cell.")
else:
    x_label = f"{TIME_UNIT.capitalize()}s Elapsed"
    x_vals = list(range(num_steps + 1))

    # Chart 1: reproductive-status distribution over time. Reconstructs
    # the undivided distribution EXACTLY from Step 7b's own
    # orig_trajectory + tagged_trajectory (no re-simulation, no
    # second grouping mechanism).
    from cow.states import repro_status
    states = states.assign(_status=repro_status(states['dp'].values))
    flag_values = sorted(states['_status'].unique())
    fig1 = go.Figure()
    for flag in flag_values:
        mask = (states['_status'] == flag).values
        y = [(o + t)[mask].sum() for o, t in zip(orig_trajectory, tagged_trajectory)]
        fig1.add_trace(go.Scatter(x=x_vals, y=y, mode='lines', stackgroup='one', name=flag,
                                   hovertemplate=f'{TIME_UNIT.capitalize()} %{{x}}<br>%{{y:.3f}}<extra></extra>'))
    fig1.update_layout(title=f"Herd Distribution by Flag over One Year ({TIME_UNIT} resolution, "
                               f"{num_steps} steps)",
                        xaxis_title=x_label, yaxis_title="Probability mass",
                        template='plotly_white', height=420)
    fig1.show()

    # Chart 2: replacement vs. regular population share over time --
    # same replacement mechanism as Step 7b/farm_simulator.py, no new
    # definition of "replaced".
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=x_vals, y=[o.sum() for o in orig_trajectory],
                               mode='lines', name='Regular (never replaced)',
                               hovertemplate=f'{TIME_UNIT.capitalize()} %{{x}}<br>%{{y:.3f}}<extra></extra>'))
    fig2.add_trace(go.Scatter(x=x_vals, y=[t.sum() for t in tagged_trajectory],
                               mode='lines', name='Replaced at least once',
                               hovertemplate=f'{TIME_UNIT.capitalize()} %{{x}}<br>%{{y:.3f}}<extra></extra>'))
    fig2.update_layout(title=f"Replacement-Aware Population Split ({TIME_UNIT} resolution)",
                        xaxis_title=x_label, yaxis_title="Probability mass",
                        template='plotly_white', height=420)
    fig2.show()